# LaTeX Table Cell Coloring

Color-code numeric cells in LaTeX tables by their values. Three normalization modes: **column**, **row**, **global**.

```python
# Quick start
print(color_by_column(r"""
Model A & 85.2 & 72.1 \\
Model B & 91.3 & 68.4 \\
""", max_tint=40))
```

Handles `\midrule`, `\textbf{...}`, comment lines (`%`), already-colored tables, and auto-detects where numbers start.

In [ ]:
import re
import warnings

# ── Color Presets ────────────────────────────────────────────────────────────
COLORS = {
    "mygreen":  "#28A745",   # Bootstrap success green
    "myred":    "#DC3545",   # Bootstrap danger red
    "myblue":   "#007BFF",   # Bootstrap primary blue
    "myorange": "#FD7E14",   # Bootstrap orange
}

# ── Internal Helpers ─────────────────────────────────────────────────────────

_STRUCTURAL_RE = re.compile(
    r'^\s*\\(midrule|hline|toprule|bottomrule|cmidrule|addlinespace)\b'
)

def _is_structural(line):
    """True for LaTeX structural commands that should pass through unchanged."""
    core = line.strip()
    # Strip trailing \\
    if core.endswith('\\\\'):
        core = core[:-2].strip()
    elif core.endswith('\\'):
        core = core[:-1].strip()
    return bool(_STRUCTURAL_RE.match(core))

def _strip_cellcolor(text):
    """Remove existing \\cellcolor{...} commands (enables recoloring)."""
    return re.sub(r'\\cellcolor\{[^}]*\}\s*', '', text)

def _extract_number(text):
    """Parse a float from text that may contain LaTeX wrappers.
    Returns float or None. Treats '-' and empty strings as non-numeric.
    """
    s = text.strip()
    if not s or s == '-':
        return None
    try:
        return float(s)
    except ValueError:
        pass
    # Fallback: extract number from LaTeX wrappers like \textbf{85.45}
    m = re.search(r'[-+]?\d*\.?\d+', s)
    if m:
        try:
            return float(m.group())
        except ValueError:
            pass
    return None

def _is_strict_float(text):
    """True if text is a plain float (no regex fallback). Used for auto-detect."""
    s = text.strip()
    if not s or s == '-':
        return False
    try:
        float(s)
        return True
    except ValueError:
        return False

def _detect_num_start(data_rows):
    """Auto-detect the first column where >=50% of data rows have a plain float.
    data_rows: list of lists of stripped cell strings.
    """
    if not data_rows:
        return 0
    num_cols = max(len(r) for r in data_rows)
    n_rows = len(data_rows)
    for c in range(num_cols):
        numeric = sum(1 for row in data_rows if c < len(row) and _is_strict_float(row[c]))
        if n_rows > 0 and numeric / n_rows >= 0.5:
            return c
    return 0

# ── Core Function ────────────────────────────────────────────────────────────

def color_latex_cells(
    table,
    norm="column",
    num_start=None,
    color="mygreen",
    max_tint=30,
    higher_is_better=True,
):
    """Add \\cellcolor tinting to numeric cells in a LaTeX table.

    Args:
        table: LaTeX table rows as a string (between \\begin{tabular} and \\end{tabular}).
               Use r\"\"\"...\"\"\" raw strings to avoid \\textbf becoming a tab.
        norm: \"column\" (per-column), \"row\" (per-row), or \"global\" (whole table).
        num_start: 0-indexed column where numbers begin. None = auto-detect.
        color: LaTeX color name (default \"mygreen\"). Must be defined in your preamble.
        max_tint: Maximum tint percentage 0-100 (default 30).
        higher_is_better: If True, higher values get more intense color.
                          Set False for metrics like FID.

    Returns:
        Colored LaTeX table rows string.
    """
    assert norm in ("column", "row", "global"), f"norm must be 'column', 'row', or 'global', got '{norm}'"

    # Warn about Python string escaping issues
    if '\t' in table or '\x0b' in table:
        warnings.warn(
            "Input contains tab/escape characters — likely from \\textbf or similar "
            "in a non-raw string. Use r\"\"\"...\"\"\" for your LaTeX input.",
            stacklevel=2,
        )

    # ── Phase 1: Parse ──────────────────────────────────────────────────────
    lines = table.strip().splitlines()
    # Each entry: ('data', row_index, cells_text_list, cells_number_list)
    #         or: ('passthrough', original_line)
    parsed = []
    data_rows_text = []    # list of lists of cell strings (cleaned)
    data_rows_nums = []    # list of lists of float|None

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        # Comment lines: pass through unchanged
        if stripped.startswith('%'):
            parsed.append(('passthrough', stripped))
            continue

        # Structural lines (\midrule, \hline, etc.): pass through
        if _is_structural(stripped):
            parsed.append(('passthrough', stripped))
            continue

        # Data row: strip trailing \\
        content = stripped
        has_terminator = False
        if content.endswith('\\\\'):
            content = content[:-2].rstrip()
            has_terminator = True
        elif content.endswith('\\'):
            content = content[:-1].rstrip()
            has_terminator = True

        # Split on & and clean each cell
        raw_cells = content.split('&')
        cells_text = [_strip_cellcolor(c).strip() for c in raw_cells]

        # Trim trailing empty cells (handles trailing & before \\)
        while len(cells_text) > 1 and cells_text[-1] == '':
            cells_text.pop()

        # Extract numbers
        cells_nums = [_extract_number(c) for c in cells_text]

        row_idx = len(data_rows_text)
        data_rows_text.append(cells_text)
        data_rows_nums.append(cells_nums)
        parsed.append(('data', row_idx))

    if not data_rows_text:
        return table  # Nothing to process

    # Pad all data rows to the same column count
    num_cols = max(len(r) for r in data_rows_text)
    for i in range(len(data_rows_text)):
        while len(data_rows_text[i]) < num_cols:
            data_rows_text[i].append('')
            data_rows_nums[i].append(None)

    # ── Phase 2: Detect num_start ───────────────────────────────────────────
    if num_start is None:
        # Use strict float parsing (no regex) to avoid false positives
        # on model names like "Stable Diffusion 1.4"
        strict_rows = []
        for row_text in data_rows_text:
            strict_rows.append(row_text)
        num_start = _detect_num_start(strict_rows)

    if num_start >= num_cols:
        return table  # No numeric columns to color

    # Re-extract numbers only for columns >= num_start using the regex fallback
    # (auto-detect used strict parsing; now we allow LaTeX-wrapped numbers)
    for i, row_text in enumerate(data_rows_text):
        for c in range(num_start, num_cols):
            data_rows_nums[i][c] = _extract_number(row_text[c])

    # ── Phase 3: Compute normalization stats ────────────────────────────────
    n_data_rows = len(data_rows_text)
    n_num_cols = num_cols - num_start

    if norm == "column":
        col_stats = []  # list of (min, max, range) per numeric column
        for c_off in range(n_num_cols):
            c = num_start + c_off
            vals = [data_rows_nums[r][c] for r in range(n_data_rows) if data_rows_nums[r][c] is not None]
            if vals:
                mn, mx = min(vals), max(vals)
                col_stats.append((mn, mx, mx - mn))
            else:
                col_stats.append((0, 0, 0))

    elif norm == "row":
        row_stats = []  # list of (min, max, range) per data row
        for r in range(n_data_rows):
            vals = [data_rows_nums[r][c] for c in range(num_start, num_cols) if data_rows_nums[r][c] is not None]
            if vals:
                mn, mx = min(vals), max(vals)
                row_stats.append((mn, mx, mx - mn))
            else:
                row_stats.append((0, 0, 0))

    elif norm == "global":
        all_vals = []
        for r in range(n_data_rows):
            for c in range(num_start, num_cols):
                v = data_rows_nums[r][c]
                if v is not None:
                    all_vals.append(v)
        if all_vals:
            g_mn, g_mx = min(all_vals), max(all_vals)
            g_stats = (g_mn, g_mx, g_mx - g_mn)
        else:
            g_stats = (0, 0, 0)

    # ── Phase 4: Apply colors ───────────────────────────────────────────────
    colored_rows = []
    for r in range(n_data_rows):
        row_out = list(data_rows_text[r])
        for c in range(num_start, num_cols):
            val = data_rows_nums[r][c]
            if val is None:
                continue

            c_off = c - num_start
            if norm == "column":
                mn, mx, rng = col_stats[c_off]
            elif norm == "row":
                mn, mx, rng = row_stats[r]
            else:  # global
                mn, mx, rng = g_stats

            if rng == 0:
                continue  # All values identical in this group

            normalized = (val - mn) / rng
            if not higher_is_better:
                normalized = 1.0 - normalized
            tint = int(round(normalized * max_tint))

            if tint > 0:
                row_out[c] = f"\\cellcolor{{{color}!{tint}}} {row_out[c]}"

        colored_rows.append(row_out)

    # ── Phase 5: Reassemble ─────────────────────────────────────────────────
    output_lines = []
    for entry in parsed:
        if entry[0] == 'passthrough':
            output_lines.append(entry[1])
        else:  # 'data'
            row_idx = entry[1]
            output_lines.append(' & '.join(colored_rows[row_idx]) + ' \\\\')

    return '\n'.join(output_lines)


# ── Convenience Wrappers ─────────────────────────────────────────────────────

def color_by_column(table, **kwargs):
    """Color cells normalized per-column. See color_latex_cells for args."""
    return color_latex_cells(table, norm="column", **kwargs)

def color_by_row(table, **kwargs):
    """Color cells normalized per-row. See color_latex_cells for args."""
    return color_latex_cells(table, norm="row", **kwargs)

def color_by_global(table, **kwargs):
    """Color cells normalized across the entire table. See color_latex_cells for args."""
    return color_latex_cells(table, norm="global", **kwargs)

## Reference

### LaTeX Preamble

Add to your preamble:

```latex
\usepackage[table]{xcolor}
\usepackage{colortbl}

% Define your colors (pick the ones you need):
\definecolor{mygreen}{HTML}{28A745}
\definecolor{myred}{HTML}{DC3545}
\definecolor{myblue}{HTML}{007BFF}
\definecolor{myorange}{HTML}{FD7E14}
```

### Tips

- **Always use raw strings** for LaTeX input: `r"""..."""` not `"""..."""`. Otherwise `\textbf` becomes a tab + `extbf`, `\newline` becomes an actual newline, etc.
- **Already-colored tables** can be passed in directly — existing `\cellcolor{...}` is stripped and re-applied.
- **Structural commands** (`\midrule`, `\hline`, `\toprule`, `\bottomrule`, `\cmidrule`, `\addlinespace`) pass through unchanged.
- **Comment lines** (starting with `%`) pass through unchanged.
- Use `higher_is_better=False` for metrics like FID where lower is better.
- `num_start` is auto-detected if omitted. Override it if the auto-detection gets it wrong.

### Color Presets

| Name | Hex | Usage |
|------|-----|-------|
| `mygreen` | `#28A745` | Default. Good for accuracy/performance metrics |
| `myred` | `#DC3545` | Good for error rates, bias scores |
| `myblue` | `#007BFF` | Neutral alternative |
| `myorange` | `#FD7E14` | Warm alternative |

## Examples

In [ ]:
# Column normalization (each column independently)
demo_table = r"""
\textbf{Base Model (SDXL)} & 86.21 & 78.21 & 62.5 & 62.97 \\
\midrule
\textbf{Prompt Revision} & & & & \\
\hspace{1em}DALL-E 3 Rewrite & 85.36 & 78.01 & 66.49 & 59.93 \\
\hspace{1em}GPT5 Translate & 85.93 & 78.12 & 69.27 & 62.69 \\
\textbf{Ours} & & & & \\
\hspace{1em}Dialect Learning & 85.45 & 78.08 & 85.99 & 82.43 \\
"""

print(color_by_column(demo_table, num_start=1, max_tint=40))

In [ ]:
# Row normalization (each row independently)
print(color_by_row(demo_table, num_start=1, max_tint=40))

In [ ]:
# Global normalization (all numeric cells together)
print(color_by_global(demo_table, num_start=1, max_tint=40, color="myred"))

In [ ]:
# Recoloring: pass already-colored output back in
colored_once = color_by_column(demo_table, num_start=1, max_tint=40)
print("--- First pass (column, tint=40) ---")
print(colored_once)
print()

# Now recolor with different settings — old \cellcolor is stripped automatically
recolored = color_by_global(colored_once, num_start=1, max_tint=60, color="myred")
print("--- Recolored (global, tint=60, red) ---")
print(recolored)